In [ ]:
import os
import re
import gc
import math
import string
import numpy as np
import pandas as pd

from collections import Counter
from dataclasses import dataclass

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

# Optional (stylometry extras)
import nltk

# Download punkt safely for Kaggle
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt")

THEME = "#5B8DEF"   # Single color theme
BG = "#0b0f19"      # dark-ish background for dashboard look (optional)
FG = "#E8EEF9"      # foreground text color
GRID = "#233044"

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.edgecolor"] = GRID
plt.rcParams["axes.labelcolor"] = FG
plt.rcParams["xtick.color"] = FG
plt.rcParams["ytick.color"] = FG
plt.rcParams["text.color"] = FG

sns.set_theme(style="darkgrid")
sns.set(rc={
    "axes.facecolor": BG,
    "figure.facecolor": BG,
    "grid.color": GRID
})

In [ ]:
df = pd.read_csv("AI_Human.csv")
print(df.shape)
df.head()

In [ ]:
# Validate columns
required_cols = {"text", "generated"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}. Found columns: {list(df.columns)}")

df = df[["text", "generated"]].copy()
df["text"] = df["text"].astype(str)
df["generated"] = df["generated"].astype(int)

# Drop empty-like rows
df["text"] = df["text"].str.strip()
df = df[df["text"].str.len() > 0].dropna()

print("After cleanup:", df.shape)
df["generated"].value_counts()

In [ ]:
counts = df["generated"].value_counts().sort_index()
labels = ["Human (0)", "AI (1)"]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, counts.values, color=THEME, edgecolor="white")
ax.set_title("Class Distribution", color=FG)
ax.set_ylabel("Count", color=FG)
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height(), f"{int(b.get_height()):,}",
            ha="center", va="bottom", color=FG)
plt.show()

In [ ]:
def normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def basic_clean(text: str) -> str:
    # Clean version for TF-IDF / BoW
    text = text.lower()
    text = normalize_whitespace(text)
    return text

df["text_raw"] = df["text"].apply(normalize_whitespace)
df["text_clean"] = df["text_raw"].apply(basic_clean)

df[["text_raw", "text_clean", "generated"]].head()

In [ ]:
def word_count(text: str) -> int:
    return len(text.split())

df["char_len"] = df["text_raw"].str.len()
df["word_len"] = df["text_raw"].apply(word_count)

df.groupby("generated")[["char_len", "word_len"]].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.kdeplot(data=df, x="word_len", hue="generated", ax=axes[0],
            palette=[FG, THEME], common_norm=False)
axes[0].set_title("Word Length Distribution (KDE)", color=FG)
axes[0].set_xlabel("Words", color=FG)

sns.kdeplot(data=df, x="char_len", hue="generated", ax=axes[1],
            palette=[FG, THEME], common_norm=False)
axes[1].set_title("Character Length Distribution (KDE)", color=FG)
axes[1].set_xlabel("Characters", color=FG)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

STOP = set(ENGLISH_STOP_WORDS)

def stopword_ratio(text: str) -> float:
    tokens = re.findall(r"[a-zA-Z']+", text.lower())
    if not tokens:
        return 0.0
    sw = sum(t in STOP for t in tokens)
    return sw / len(tokens)

df["stopword_ratio"] = df["text_raw"].apply(stopword_ratio)

fig, ax = plt.subplots(figsize=(10, 4))
sns.violinplot(data=df, x="generated", y="stopword_ratio",
               palette=[FG, THEME], ax=ax)
ax.set_title("Stopword Ratio by Class", color=FG)
ax.set_xlabel("generated (0=Human, 1=AI)", color=FG)
ax.set_ylabel("Stopword ratio", color=FG)
plt.show()

In [ ]:
def top_words(texts, n=20):
    tokens = []
    for t in texts:
        toks = re.findall(r"[a-zA-Z']+", t.lower())
        tokens.extend([x for x in toks if x not in STOP and len(x) > 2])
    return Counter(tokens).most_common(n)

top_h = top_words(df.loc[df.generated==0, "text_raw"].head(20000), n=20)
top_a = top_words(df.loc[df.generated==1, "text_raw"].head(20000), n=20)

top_h_df = pd.DataFrame(top_h, columns=["word", "count"])
top_a_df = pd.DataFrame(top_a, columns=["word", "count"])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.barplot(data=top_h_df, y="word", x="count", color=THEME, ax=axes[0])
axes[0].set_title("Top Words (Human) — sample", color=FG)
sns.barplot(data=top_a_df, y="word", x="count", color=THEME, ax=axes[1])
axes[1].set_title("Top Words (AI) — sample", color=FG)

plt.tight_layout()
plt.show()

In [ ]:
def top_ngrams_tfidf(texts, ngram_range=(2,2), top_n=20, max_features=200000):
    vec = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features, min_df=2)
    X = vec.fit_transform(texts)
    scores = np.asarray(X.mean(axis=0)).ravel()
    idx = np.argsort(scores)[::-1][:top_n]
    inv = {v:k for k,v in vec.vocabulary_.items()}
    return [(inv[i], scores[i]) for i in idx if i in inv]

human_phrases = top_ngrams_tfidf(df.loc[df.generated==0, "text_clean"].head(50000), (2,2), 20)
ai_phrases = top_ngrams_tfidf(df.loc[df.generated==1, "text_clean"].head(50000), (2,2), 20)

pd.DataFrame(human_phrases, columns=["bigram", "avg_tfidf"]).head(10), pd.DataFrame(ai_phrases, columns=["bigram", "avg_tfidf"]).head(10)

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize

def safe_sent_tokenize(text: str):
    # nltk sentence tokenizer can fail rarely on very weird input; guard it
    try:
        sents = sent_tokenize(text)
        return [s.strip() for s in sents if s.strip()]
    except Exception:
        # fallback: naive split
        return [s.strip() for s in re.split(r"[.!?]+", text) if s.strip()]

def tokens_alpha(text: str):
    return re.findall(r"[A-Za-z']+", text)

def type_token_ratio(tokens):
    if not tokens:
        return 0.0
    return len(set(tokens)) / len(tokens)

def avg_word_len(tokens):
    if not tokens:
        return 0.0
    return float(np.mean([len(t) for t in tokens]))

def sentence_word_lengths(text: str):
    sents = safe_sent_tokenize(text)
    if not sents:
        return (0.0, 0.0, 0.0, 0)
    lens = [len(tokens_alpha(s)) for s in sents]
    lens = [l for l in lens if l > 0]
    if not lens:
        return (0.0, 0.0, 0.0, len(sents))
    return (float(np.mean(lens)), float(np.std(lens)), float(np.max(lens)), len(lens))

def repeated_ngram_ratio(text: str, n=3):
    toks = [t.lower() for t in tokens_alpha(text)]
    if len(toks) < n:
        return 0.0
    grams = [" ".join(toks[i:i+n]) for i in range(len(toks)-n+1)]
    if not grams:
        return 0.0
    c = Counter(grams)
    repeated = sum(v for v in c.values() if v > 1)
    return repeated / len(grams)

def punctuation_rate(text: str):
    if not text:
        return 0.0
    punct = sum(ch in string.punctuation for ch in text)
    return punct / len(text)

def uppercase_ratio(text: str):
    if not text:
        return 0.0
    upp = sum(ch.isupper() for ch in text)
    alpha = sum(ch.isalpha() for ch in text)
    return (upp / alpha) if alpha else 0.0

def stylometry_features(text: str) -> dict:
    text = normalize_whitespace(text)
    toks = [t.lower() for t in tokens_alpha(text)]
    ttr = type_token_ratio(toks)

    sent_mean, sent_std, sent_max, n_sents = sentence_word_lengths(text)

    feats = {
        "word_len": len(text.split()),
        "char_len": len(text),
        "n_sents": n_sents,
        "sent_len_mean": sent_mean,
        "sent_len_std": sent_std,
        "sent_len_max": sent_max,
        "ttr": ttr,
        "avg_word_len": avg_word_len(toks),
        "repeated_trigram_ratio": repeated_ngram_ratio(text, n=3),
        "punct_rate": punctuation_rate(text),
        "uppercase_ratio": uppercase_ratio(text),
        "comma_per_1000c": (text.count(",") / max(1, len(text))) * 1000.0,
        "semicolon_per_1000c": (text.count(";") / max(1, len(text))) * 1000.0,
        "exclam_per_1000c": (text.count("!") / max(1, len(text))) * 1000.0,
        "question_per_1000c": (text.count("?") / max(1, len(text))) * 1000.0,
    }
    return feats

# Compute stylometry (vectorized-ish via apply; for 500k rows it's still OK but can take time)
# If you want speed, you can sample or use swifter; here we keep it Kaggle-safe.
sty = df["text_raw"].apply(stylometry_features)
sty_df = pd.DataFrame(list(sty))
sty_df["generated"] = df["generated"].values
sty_df.head()

In [ ]:
feat_cols = [c for c in sty_df.columns if c != "generated"]

summary = sty_df.groupby("generated")[feat_cols].mean().T
summary.columns = ["Human_mean", "AI_mean"]
summary["diff_AI_minus_Human"] = summary["AI_mean"] - summary["Human_mean"]
summary.sort_values("diff_AI_minus_Human", ascending=False).head(15)

In [ ]:
# Plot a subset of the most discriminative stylometry means
topk = 10
top_feats = summary["diff_AI_minus_Human"].abs().sort_values(ascending=False).head(topk).index.tolist()

plot_df = sty_df.melt(id_vars="generated", value_vars=top_feats,
                      var_name="feature", value_name="value")

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=plot_df, x="feature", y="value", hue="generated",
            palette=[FG, THEME], ax=ax, estimator=np.mean, errorbar=None)
ax.set_title("Stylometry Feature Means (Human vs AI)", color=FG)
ax.set_xlabel("")
ax.set_ylabel("Mean value", color=FG)
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()